# Daily Sales Between Squarespace and TikTok Shop and Overall Total Sales
Merging Squarespace and TikTok Shop daily sales 

## Load Squarespace and TikTok Shops Data

In [1]:
import pandas as pd

In [2]:
ss = pd.read_csv("../data/raw/synthetic_squarespace_sales_2024_WEIGHTED.csv")
tts = pd.read_csv("../data/raw/synthetic_tiktok_shop_sales_2024_DOMINANT.csv")

## Preview of Squarespace and TikTok Shop Sales Data

In [3]:
ss.head()

,order_id,order_date,product_name,quantity,unit_price,gross_sales,discount_amount,refund_amount,net_sales,state
0,SS3000,2024-01-01,My Hero Academia Backpack,1,48,48,0.00,0.0,48.00,Florida
1,SS3001,2024-01-01,One Piece Figure,1,35,35,6.42,0.0,28.58,Illinois
2,SS3002,2024-01-01,Attack on Titan T-Shirt,1,28,28,0.00,0.0,28.00,Texas
3,SS3003,2024-01-01,Anime Mystery Box,1,60,60,6.16,0.0,53.84,Massachusetts
4,SS3004,2024-01-01,Anime Mystery Box,1,60,60,0.00,0.0,60.00,California


In [4]:
tts.head()

,order_date,total_sales
0,2024-01-01,2996.84
1,2024-01-02,3167.73
2,2024-01-03,4251.54
3,2024-01-04,4285.11
4,2024-01-06,2033.86


## Merging Squarespace and TikTok Shop Datasets

In [5]:
merged_sales = ss.merge(
      tts
    , on="order_date"
    , how="left"
)

### Transforming NULL TikTok Sales Values
Nulls become 0 so that calculations can be ran normally 

In [6]:
merged_sales["total_tts_sales_clean"]=merged_sales["total_sales"].fillna(0)

### Preview of Merged Sales Data

In [7]:
merged_sales.head()

,order_id,order_date,product_name,quantity,unit_price,gross_sales,discount_amount,refund_amount,net_sales,state,total_sales,total_tts_sales_clean
0,SS3000,2024-01-01,My Hero Academia Backpack,1,48,48,0.00,0.0,48.00,Florida,2996.84,2996.84
1,SS3001,2024-01-01,One Piece Figure,1,35,35,6.42,0.0,28.58,Illinois,2996.84,2996.84
2,SS3002,2024-01-01,Attack on Titan T-Shirt,1,28,28,0.00,0.0,28.00,Texas,2996.84,2996.84
3,SS3003,2024-01-01,Anime Mystery Box,1,60,60,6.16,0.0,53.84,Massachusetts,2996.84,2996.84
4,SS3004,2024-01-01,Anime Mystery Box,1,60,60,0.00,0.0,60.00,California,2996.84,2996.84


### Daily Total Squarespace Sales: Group By order_date
Originally, there were multiple net_sales on given day resulting in multiple records for one day. The goal was to combine total sales into each day.

In [8]:
merged_sales.groupby("order_date")["net_sales"].sum()

order_date
2024-01-01    1000.97
2024-01-02     612.93
2024-01-03    1060.89
2024-01-04    1505.34
2024-01-05     468.37
               ...   
2024-12-27    1175.60
2024-12-28    1043.54
2024-12-29     864.06
2024-12-30    1308.71
2024-12-31     399.00
Name: net_sales, Length: 366, dtype: float64

## Created New Aggreated Columns
Showing Daily Sales between Squarespace and TikTok Shop. While Squarespace sales needed sum, I only needed to select for one of the repeated values of total_tts_sales_clean.

In [9]:
clean_merged_sales=merged_sales.groupby("order_date").agg(
    total_ss_sales=("net_sales", "sum"), 
    total_tts_sales_clean=("total_tts_sales_clean", "first")
)

## Created a new column as total for Daily Squarespace and Tiktok Sales 

In [10]:
clean_merged_sales["total_sales_overall"] = clean_merged_sales["total_ss_sales"] + clean_merged_sales["total_tts_sales_clean"]

### Fixing the order_date column to not be the index
Takes the transformed DataFrame and restores order_date as a normal column

In [11]:
clean_merged_sales = clean_merged_sales.reset_index().sort_values(
    by="order_date", 
    ascending=True
)

## Final DataFrame showing Daily Squarespace and TikTok Sales and total overall sales
Showing only the relevant columns 

In [12]:
clean_merged_sales

,order_date,total_ss_sales,total_tts_sales_clean,total_sales_overall
0,2024-01-01,1000.97,2996.84,3997.81
1,2024-01-02,612.93,3167.73,3780.66
2,2024-01-03,1060.89,4251.54,5312.43
3,2024-01-04,1505.34,4285.11,5790.45
4,2024-01-05,468.37,0.00,468.37
...,...,...,...,...
361,2024-12-27,1175.60,0.00,1175.60
362,2024-12-28,1043.54,2531.02,3574.56
363,2024-12-29,864.06,4137.24,5001.30
364,2024-12-30,1308.71,0.00,1308.71


## Breakdown of the Validation Results
2 datasets were left join(ed) onto the Squarespace dataset. All records were grouped by order_date, and each sales platform's daily sales were displayed in their own column. Lastly, a new coulumn for total sales was created. These results validate the findings from my orgiginal SQL analysis of daily sales between Squarespace and TikTok Shop.  